In [1]:

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from torch.nn.utils.rnn import pad_sequence
import torch.nn.functional as F
import pandas as pd
import copy
from tqdm import tqdm
import tiktoken

In [ ]:
train_df = pd.read_csv('train.csv').dropna()
test_df = pd.read_csv('validation.csv')

In [ ]:
class TinyStories(Dataset):
    def __init__(self, df):
        self.df = df
        self.tokenizer = tiktoken.get_encoding("r50k_base")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.df['text'].iloc[idx]
        tokens = self.tokenizer.encode(text)
        input_ids = torch.tensor(tokens[:-1])
        target = torch.tensor(tokens[1:])
        return input_ids, target

In [ ]:
def collate_fn(batch):
    
    input_ids, target = zip(*batch)
    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=0)
    target = pad_sequence(target, batch_first=True, padding_value=0)

    b, s = input_ids.shape

    attention_mask = (input_ids == 0)

    causal_mask = torch.triu(
        torch.ones(s, s), diagonal=1
    ).bool()

    return input_ids, attention_mask, causal_mask, target

In [ ]:

train_dataset = TinyStories(df=train_df)
test_dataset = TinyStories(df=test_df)

In [ ]:
train_dataloader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
)

In [ ]:
input_ids, attention_mask, causal_mask, target = next(iter(train_dataloader))

In [ ]:
scaler = GradScaler('mps')

def train_step(model, optimizer, dataloader, epoch, device="mps"):
    model.train()
    train_loss = 0

    # progress_bar = tqdm(dataloader, desc=f"Epoch {epoch}", dynamic_ncols=True)
    
    for batch_idx, (input_ids, attention_mask, causal_mask, target) in enumerate(dataloader):
        input_ids, attention_mask, causal_mask, target = input_ids.to(device), attention_mask.to(device), causal_mask.to(device), target.to(device)

        with autocast('mps'):
            logits = model(input_ids)
            loss = F.cross_entropy(logits.permute(0,2,1), target, ignore_index=0, label_smoothing=0.1)

            batch_loss = loss.item()
            train_loss += batch_loss

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

        # avg_loss = train_loss / (batch_idx + 1)
        # progress_bar.set_postfix(batch_loss=f"{batch_loss:.4f}", avg_loss=f"{avg_loss:.4f}")

        del input_ids, attention_mask, causal_mask, loss, logits

    avg_loss = train_loss / len(dataloader)

    print(f"Epoch {epoch} | Train Loss: {avg_loss:.4f}")
    return avg_loss



def test_step(model, epoch, dataloader, device="mps"):
    model.eval()
    test_loss, total_tokens = 0, 0

    # progress_bar = tqdm(dataloader, desc=f"Epoch {epoch} (Eval)", dynamic_ncols=True)

    with torch.no_grad():
        for batch_idx, (input_ids, attention_mask, causal_mask, target) in enumerate(dataloader):
            input_ids, attention_mask, causal_mask, target = input_ids.to(device), attention_mask.to(device), causal_mask.to(device), target.to(device)

            with autocast('mps'):
                logits = model(input_ids)
                loss = F.cross_entropy(logits.permute(0,2,1), target, ignore_index=0, label_smoothing=0.1, reduction='sum')
    
                batch_loss = loss.item()
                test_loss += batch_loss
                batch_tokens = (~attention_mask).sum().item()
                total_tokens += batch_tokens

            # batch_ppl = torch.exp(torch.tensor(batch_loss / batch_tokens))
            # progress_bar.set_postfix(batch_ppl=f"{batch_ppl:.4f}")
    
            del input_ids, attention_mask, causal_mask, loss, logits

    avg_loss = test_loss / total_tokens
    avg_nll = test_loss / total_tokens
    ppl = torch.exp(torch.tensor(avg_nll))

    print(f"Epoch {epoch} | Test Loss: {avg_loss:.4f} | Test Preplexity: {ppl:.4f}")
    return avg_loss, ppl

In [ ]:
from transformer.transformer import Transformer

ChatGPT6 = Transformer()
ChatGPT6 = ChatGPT6.to("mps")



In [ ]:
current = 13
num_epochs = 1

start = 1 + current
epochs = num_epochs + current + 1

print(f"Running from {start} to {epochs-1}")

model = nn.DataParallel(ChatGPT6)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01, betas=(0.9, 0.95))

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=5,   
    eta_min=3e-5       
)

torch.manual_seed(42)
torch.cuda.manual_seed(42)

print("Ready to train!!")

for epoch in range(start, epochs):
    train_step(
        model=model,
        optimizer=optimizer,
        epoch=epoch,
        dataloader=train_dataloader
    )

    test_step(
        model=model,
        epoch=epoch,
        dataloader=test_dataloader,
    )
    # scheduler.step()
    checkpoint = {
        'model_state_dict' : model.state_dict(),
        'optimizer_state_dict' : optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict()
    }
    torch.save(obj=checkpoint, f=f"BlockFormer_{epoch}_epochs.pth")

Running from 14 to 14
Ready to train!!


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, mps:0 and cpu!